In [ ]:
def extract_content_from_pdf(pdf_path, output_dir=None):
    """
    从PDF文件中提取文本和图像。

    参数:
        pdf_path (str): PDF文件的路径
        output_dir (str, 可选): 提取的图像保存的目录

    返回:
        Tuple[List[Dict], List[Dict]]: 文本数据和图像数据
    """
    # 如果未提供输出目录，则创建一个临时目录用于存储图像
    temp_dir = None
    if output_dir is None:
        temp_dir = tempfile.mkdtemp()
        output_dir = temp_dir
    else:
        os.makedirs(output_dir, exist_ok=True)

    text_data = []  # 用于存储提取的文本数据的列表
    image_paths = []  # 用于存储提取的图像路径的列表

    print(f"Extracting content from {pdf_path}...")

    try:
        with fitz.open(pdf_path) as pdf_file:
            # 遍历PDF中的每一页
            for page_number in range(len(pdf_file)):
                page = pdf_file[page_number]

                # 从页面提取文本
                text = page.get_text().strip()
                if text:
                    text_data.append(
                        {
                            "content": text,
                            "metadata": {
                                "source": pdf_path,
                                "page": page_number + 1,
                                "type": "text",
                            },
                        }
                    )

                # 从页面提取图像
                image_list = page.get_images(full=True)
                for img_index, img in enumerate(image_list):
                    xref = img[0]  # 图像的XREF
                    base_image = pdf_file.extract_image(xref)

                    if base_image:
                        image_bytes = base_image["image"]
                        image_ext = base_image["ext"]

                        # 将图像保存到输出目录
                        img_filename = (
                            f"page_{page_number + 1}_img_{img_index + 1}.{image_ext}"
                        )
                        img_path = os.path.join(output_dir, img_filename)

                        with open(img_path, "wb") as img_file:
                            img_file.write(image_bytes)

                        image_paths.append(
                            {
                                "path": img_path,
                                "metadata": {
                                    "source": pdf_path,
                                    "page": page_number + 1,
                                    "image_index": img_index + 1,
                                    "type": "image",
                                },
                            }
                        )

        print(f"Extracted {len(text_data)} text segments and {len(image_paths)} images")
        return text_data, image_paths

    except Exception as e:
        print(f"Error extracting content: {e}")
        if temp_dir and os.path.exists(temp_dir):
            shutil.rmtree(temp_dir)
        raise
